<a href="https://colab.research.google.com/github/aparna-2001/GATE_PPI/blob/main/negatome_homo_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
negatome_raw = pd.read_csv(
    '/content/combined_stringent.txt',
    sep='\t',
    header=None,
    names=['protein1', 'protein2'],
    encoding='latin-1'
)

print(f"Negatome raw pairs: {len(negatome_raw):,}")
print(negatome_raw.head())

negatome_proteins_raw = set(negatome_raw['protein1']) | set(negatome_raw['protein2'])
print(f"\nUnique proteins in raw Negatome: {len(negatome_proteins_raw):,}")

Negatome raw pairs: 6,136
  protein1 protein2
0   Q6ZNK6   Q9Y4K3
1   Q9NR31   Q15797
2   P11627   P53986
3   P33176   Q96EK5
4   Q9NPY3   P02745

Unique proteins in raw Negatome: 3,215


In [ ]:
import requests
import time

test_proteins = list(negatome_proteins_raw)[:5]
print(f"Testing on: {test_proteins}\n")

for uid in test_proteins:
    url = f"https://rest.uniprot.org/uniprotkb/{uid}.json"
    try:
        resp = requests.get(url, timeout=10)
        print(f"{uid}: status={resp.status_code}")
        if resp.status_code == 200:
            data = resp.json()
            organism = data.get('organism', {}).get('scientificName', 'UNKNOWN')
            print(f"   organism = {organism}")
        else:
            print(f"   response: {resp.text[:150]}")
    except Exception as e:
        print(f"{uid}: EXCEPTION - {type(e).__name__}: {e}")
    time.sleep(0.5)   # slower this time, 2 requests/sec, to avoid rate limiting

Testing on: ['B2PBH3', 'Q9BUQ8', 'P42768', 'P07321', 'P38431']

B2PBH3: status=200
   organism = UNKNOWN
Q9BUQ8: status=200
   organism = Homo sapiens
P42768: status=200
   organism = Homo sapiens
P07321: status=200
   organism = Mus musculus
P38431: status=200
   organism = Saccharomyces cerevisiae (strain ATCC 204508 / S288c)


In [ ]:
import json

url = "https://rest.uniprot.org/uniprotkb/B2PBH3.json"
resp = requests.get(url, timeout=10)
data = resp.json()
print(json.dumps(data, indent=2)[:1500])

{
  "entryType": "Inactive",
  "primaryAccession": "B2PBH3",
  "uniProtkbId": "B2PBH3_ECO57",
  "annotationScore": 0.0,
  "inactiveReason": {
    "inactiveReasonType": "DELETED",
    "deletedReason": "Redundant proteome"
  },
  "extraAttributes": {
    "uniParcId": "UPI0000157414"
  }
}


In [ ]:
import requests
import time
import json

def check_organism(uid):
    """
    Checks UniProt for a given accession.
    Returns dict with organism, entry_type, is_human, is_active flags.
    Handles inactive entries explicitly.
    """
    url = f"https://rest.uniprot.org/uniprotkb/{uid}.json"
    try:
        resp = requests.get(url, timeout=10)
        if resp.status_code != 200:
            return {'organism': 'NOT_FOUND', 'entry_type': 'NOT_FOUND',
                    'is_human': False, 'is_active': False}

        data = resp.json()
        entry_type = data.get('entryType', 'UNKNOWN')

        # Inactive entries have no 'organism' key at all
        if entry_type == 'Inactive':
            return {'organism': 'INACTIVE', 'entry_type': entry_type,
                     'is_human': False, 'is_active': False}

        organism = data.get('organism', {}).get('scientificName', 'UNKNOWN')
        return {
            'organism': organism,
            'entry_type': entry_type,
            'is_human': organism == 'Homo sapiens',
            'is_active': True
        }
    except Exception as e:
        return {'organism': f'ERROR: {e}', 'entry_type': 'ERROR',
                 'is_human': False, 'is_active': False}


negatome_proteins_list = list(negatome_proteins_raw)
organism_results = {}

chunk_size = 200
total = len(negatome_proteins_list)

print(f"Checking {total} proteins in chunks of {chunk_size}...\n")

for start in range(0, total, chunk_size):
    end = min(start + chunk_size, total)
    chunk = negatome_proteins_list[start:end]

    for uid in chunk:
        organism_results[uid] = check_organism(uid)
        time.sleep(0.15)   # ~6-7 requests/sec, safely under rate limits

    print(f"  Completed {end}/{total}")

    # Save progress after every chunk — never lose more than one chunk of work
    with open('/content/negatome_organism_check.json', 'w') as f:
        json.dump(organism_results, f, indent=2)

print("\nAll done. Results saved.")

Checking 3215 proteins in chunks of 200...

  Completed 200/3215
  Completed 400/3215
  Completed 600/3215
  Completed 800/3215
  Completed 1000/3215
  Completed 1200/3215
  Completed 1400/3215
  Completed 1600/3215
  Completed 1800/3215
  Completed 2000/3215
  Completed 2200/3215
  Completed 2400/3215
  Completed 2600/3215
  Completed 2800/3215
  Completed 3000/3215
  Completed 3200/3215
  Completed 3215/3215

All done. Results saved.


In [ ]:
n_human    = sum(1 for v in organism_results.values() if v['is_human'])
n_nonhuman = sum(1 for v in organism_results.values() if v['is_active'] and not v['is_human'])
n_inactive = sum(1 for v in organism_results.values() if not v['is_active'])

print(f"Human proteins:      {n_human:,}")
print(f"Non-human (active):  {n_nonhuman:,}")
print(f"Inactive/deleted:    {n_inactive:,}")
print(f"Total checked:       {len(organism_results):,}")

from collections import Counter
non_human_organisms = Counter(
    v['organism'] for v in organism_results.values()
    if v['is_active'] and not v['is_human']
)
print(f"\nTop non-human organisms found:")
for org, count in non_human_organisms.most_common(20):
    print(f"  {org}: {count}")

Human proteins:      1,175
Non-human (active):  1,793
Inactive/deleted:    247
Total checked:       3,215

Top non-human organisms found:
  Mus musculus: 495
  Saccharomyces cerevisiae (strain ATCC 204508 / S288c): 164
  Rattus norvegicus: 150
  Bos taurus: 88
  Escherichia coli (strain K12): 68
  Arabidopsis thaliana: 36
  Thermosynechococcus vestitus (strain NIES-2133 / IAM M-273 / BP-1): 33
  Gallus gallus: 33
  Drosophila melanogaster: 30
  Thermostichus vulcanus: 20
  Thermus thermophilus (strain ATCC 27634 / DSM 579 / HB8): 20
  Schizosaccharomyces pombe (strain 972 / ATCC 24843): 20
  Sus scrofa: 19
  Human immunodeficiency virus type 1: 16
  Pisum sativum: 16
  Saccharolobus solfataricus (strain ATCC 35092 / DSM 1617 / JCM 11322 / P2): 15
  Xenopus laevis: 13
  Saccharolobus shibatae (strain ATCC 51178 / DSM 5389 / JCM 8931 / NBRC 15437 / B12): 13
  Human cytomegalovirus (strain AD169): 10
  Hirudo medicinalis: 10


In [ ]:
def is_pair_human(row, organism_results):
    p1_human = organism_results.get(row['protein1'], {}).get('is_human', False)
    p2_human = organism_results.get(row['protein2'], {}).get('is_human', False)
    return p1_human and p2_human

negatome_raw['both_human'] = negatome_raw.apply(
    lambda row: is_pair_human(row, organism_results), axis=1
)

negatome_human = negatome_raw[negatome_raw['both_human']].drop(columns=['both_human']).copy()

print(f"Original Negatome pairs:  {len(negatome_raw):,}")
print(f"Human-only pairs kept:    {len(negatome_human):,}")
print(f"Pairs dropped:            {len(negatome_raw) - len(negatome_human):,}")
print(f"\nFirst 5 rows:")
print(negatome_human.head())

Original Negatome pairs:  6,136
Human-only pairs kept:    1,438
Pairs dropped:            4,698

First 5 rows:
  protein1 protein2
0   Q6ZNK6   Q9Y4K3
1   Q9NR31   Q15797
3   P33176   Q96EK5
4   Q9NPY3   P02745
7   Q99576   O43524


In [ ]:
import json
from datetime import datetime

negatome_human['label']  = 0
negatome_human['source'] = 'Negatome_combined_stringent_human'

negatome_human.to_csv('/content/negatome_human.tsv', sep='\t', index=False)
negatome_human.to_parquet('/content/negatome_human.parquet', index=False)

metadata = {
    "date_created":          datetime.today().strftime('%Y-%m-%d'),
    "source":                "Negatome 2.0 Combined-stringent, filtered to human-only",
    "original_pairs":        6136,
    "human_only_pairs":      len(negatome_human),
    "dropped_pairs":         6136 - len(negatome_human),
    "drop_reason":           "Pair excluded if either protein is non-human, inactive, or not found in UniProt",
    "unique_proteins_checked": 3215,
    "unique_proteins_human":   1175,
    "unique_proteins_nonhuman": 1793,
    "unique_proteins_inactive": 247,
    "label": 0
}

with open('/content/negatome_human_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("Saved successfully")
print(json.dumps(metadata, indent=2))

Saved successfully
{
  "date_created": "2026-06-17",
  "source": "Negatome 2.0 Combined-stringent, filtered to human-only",
  "original_pairs": 6136,
  "human_only_pairs": 1438,
  "dropped_pairs": 4698,
  "drop_reason": "Pair excluded if either protein is non-human, inactive, or not found in UniProt",
  "unique_proteins_checked": 3215,
  "unique_proteins_human": 1175,
  "unique_proteins_nonhuman": 1793,
  "unique_proteins_inactive": 247,
  "label": 0
}


In [ ]:
n_positives    = 10405
n_negatome_new = len(negatome_human)
n_random       = 25079   # unchanged, already clean

n_negatives_total = n_negatome_new + n_random
ratio = n_negatives_total / n_positives

print(f"Positives:         {n_positives:,}")
print(f"Negatome (human):  {n_negatome_new:,}")
print(f"Random negatives:  {n_random:,}")
print(f"Total negatives:   {n_negatives_total:,}")
print(f"Ratio:             {ratio:.4f}")

Positives:         10,405
Negatome (human):  1,438
Random negatives:  25,079
Total negatives:   26,517
Ratio:             2.5485


In [ ]:
# Reload positives and random negatives (unaffected, but let's be explicit)

# Loading 'positives' from parquet file
positives = pd.read_parquet('/content/positives_string.parquet')

# Loading 'random_neg_df' from parquet file
random_neg_df = pd.read_parquet('/content/negatives_random.parquet')

positives_clean = positives[['protein1', 'protein2']].copy()
positives_clean['label']  = 1
positives_clean['source'] = 'STRING_experimental'

random_clean = random_neg_df[['protein1', 'protein2', 'label', 'source']].copy()

negatome_clean = negatome_human[['protein1', 'protein2', 'label', 'source']].copy()

# Merge all three — using the CLEAN negatome this time
pairs_combined = pd.concat(
    [positives_clean, negatome_clean, random_clean],
    ignore_index=True
)

print(f"Total rows after merge: {len(pairs_combined):,}")
print(f"\nLabel distribution:")
print(pairs_combined['label'].value_counts())
print(f"\nSource distribution:")
print(pairs_combined['source'].value_counts())

Total rows after merge: 36,922

Label distribution:
label
0    26517
1    10405
Name: count, dtype: int64

Source distribution:
source
random_negative                      25079
STRING_experimental                  10405
Negatome_combined_stringent_human     1438
Name: count, dtype: int64


In [ ]:
# Check 1 — exact duplicate rows
exact_duplicates = pairs_combined.duplicated().sum()
print(f"Exact duplicate rows: {exact_duplicates}")

# Check 2 — duplicate pairs regardless of label/source
pair_duplicates = pairs_combined.duplicated(subset=['protein1', 'protein2']).sum()
print(f"Duplicate pairs (ignoring label/source): {pair_duplicates}")

# Check 3 — contradictory labels
pair_counts = pairs_combined.groupby(['protein1', 'protein2'])['label'].nunique()
contradictory = (pair_counts > 1).sum()
print(f"Contradictory labels: {contradictory}")

Exact duplicate rows: 32
Duplicate pairs (ignoring label/source): 32
Contradictory labels: 0


In [ ]:
before = len(pairs_combined)

pairs_combined = pairs_combined.drop_duplicates(
    subset=['protein1', 'protein2'],
    keep='first'
).reset_index(drop=True)

after = len(pairs_combined)

print(f"Rows before: {before:,}")
print(f"Rows removed: {before - after:,}")
print(f"Rows after: {after:,}")
print(f"\nLabel distribution after dedup:")
print(pairs_combined['label'].value_counts())
print(f"\nSource distribution after dedup:")
print(pairs_combined['source'].value_counts())

Rows before: 36,922
Rows removed: 32
Rows after: 36,890

Label distribution after dedup:
label
0    26485
1    10405
Name: count, dtype: int64

Source distribution after dedup:
source
random_negative                      25079
STRING_experimental                  10405
Negatome_combined_stringent_human     1406
Name: count, dtype: int64


In [ ]:
import json
from datetime import datetime

pairs_combined.to_csv('/content/pairs_combined.tsv', sep='\t', index=False)
pairs_combined.to_parquet('/content/pairs_combined.parquet', index=False)

metadata = {
    "date_created":         datetime.today().strftime('%Y-%m-%d'),
    "total_pairs":          len(pairs_combined),
    "positives":            int((pairs_combined['label']==1).sum()),
    "negatives":            int((pairs_combined['label']==0).sum()),
    "ratio":                round((pairs_combined['label']==0).sum() /
                                  (pairs_combined['label']==1).sum(), 4),
    "duplicates_removed":   32,
    "sources": {
        "STRING_experimental":                  int((pairs_combined['source']=='STRING_experimental').sum()),
        "Negatome_combined_stringent_human":    int((pairs_combined['source']=='Negatome_combined_stringent_human').sum()),
        "random_negative":                      int((pairs_combined['source']=='random_negative').sum())
    },
    "correction_note": "Negatome was re-filtered to human-only proteins (organism=Homo sapiens via UniProt) prior to this merge, since the original Negatome PDB-derived subset is not organism-restricted and introduced cross-species contamination (mouse, yeast, rat, viral proteins).",
    "id_space_note": "STRING and random_negative use Ensembl IDs (9606.ENSPxxx). Negatome uses UniProt IDs. Reconciliation happens in Step 5.",
    "checks_passed": {
        "exact_duplicates_before_removal":  32,
        "contradictory_labels":             0,
        "duplicates_after_removal":         0
    }
}

with open('/content/pairs_combined_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("Saved successfully")
print(json.dumps(metadata, indent=2))

Saved successfully
{
  "date_created": "2026-06-17",
  "total_pairs": 36890,
  "positives": 10405,
  "negatives": 26485,
  "ratio": 2.5454,
  "duplicates_removed": 32,
  "sources": {
    "STRING_experimental": 10405,
    "Negatome_combined_stringent_human": 1406,
    "random_negative": 25079
  },
  "correction_note": "Negatome was re-filtered to human-only proteins (organism=Homo sapiens via UniProt) prior to this merge, since the original Negatome PDB-derived subset is not organism-restricted and introduced cross-species contamination (mouse, yeast, rat, viral proteins).",
  "id_space_note": "STRING and random_negative use Ensembl IDs (9606.ENSPxxx). Negatome uses UniProt IDs. Reconciliation happens in Step 5.",
  "checks_passed": {
    "exact_duplicates_before_removal": 32,
    "contradictory_labels": 0,
    "duplicates_after_removal": 0
  }
}
